In [1]:
import numpy as np
import pandas as pd

In [2]:
nz_results=pd.DataFrame(columns=["Z score",'Win percentage','Tie percentage','Home Matches','Home Win Percentage','Away win Percentage','Sport'])

# Start of Cricket Analysis

In [3]:
# Cleaning cricket Dataset
cricket_df=pd.read_csv("icc_cwc.csv")
cricket_df=cricket_df.dropna()
# Dropping rows not neccessary for analysis
cricket_df=cricket_df.drop(["Inns","Opposition Innings","Overs","Opposition Overs"],axis=1)
# Cleaning so Opposition is just country name
cricket_df["Opposition"]=cricket_df["Opposition"].apply(lambda x: x.replace('v','').strip())
# Converting score to integer for Score and Opposition score
cricket_df["Score"]=pd.to_numeric(cricket_df["Score"].apply(lambda x: x.split("/")[0]),errors='coerce')
cricket_df["Opposition Score"]=pd.to_numeric(cricket_df["Opposition Score"].apply(lambda x: x.split("/")[0]))

In [4]:
# Make list of NZ cities so can establish when game is played on Home ground i.e. in New Zealand
nz_cricket_cities=['Auckland','Hamilton','Napier','Dunedin','Wellington','Christchurch']

In [264]:
# Getting all matches where NZ is Team 1 or Opposition
nz_cricket_df=cricket_df[(cricket_df['Team 1']=="New Zealand")|(cricket_df["Opposition"]=="New Zealand")]
# Creating new Column Home to show when game is played in NZ
nz["Home"]=nz["Ground"].isin(nz_cricket_cities)
nz_team1=nz[nz["Team 1"]=="New Zealand"]
nz_opp=nz[nz["Opposition"]=="New Zealand"]
# Number of matches where NZ as Team 1 Won
nz_win=len(nz[(nz["Team 1"]=="New Zealand")&(nz["Result"]=="won")])
# Number of matches where NZ as opposition won. Thus meaning Team 1 lost.
other_lost=len(nz[(nz["Opposition"]=="New Zealand")&(nz["Result"]=='lost')])

# Win Percentage as total number of matches won divided by total matches that NZ played
nz_cricket_win_percent=np.round((nz_win+other_lost)/(len(nz)),3)
# Percentage of matches where NZ tied
nz_tie_percent=np.round(len(nz[nz['Result']=="tied"])/len(nz),2)

In [102]:
# Computing mean score for NZ cricket scores
nz_cricket_scores=np.concatenate((nz_team1["Score"],nz_opp["Opposition Score"]))
nz_cricket_mean=nz_cricket_scores.mean()
nz_cricket_std=nz_cricket_scores.std()**2/len(nz_cricket_scores)

In [103]:
# Mean score for all teams in Dataset
total_cricket_scores=pd.DataFrame(np.concatenate((cricket_df["Score"],cricket_df["Opposition Score"])))
total_cricket_mean=total_cricket_scores.mean()
total_cricket_std=total_cricket_scores.std()**2/len(total_cricket_df)

In [276]:
# Z score for testing significance of result
cricket_z_score=np.round(pd.to_numeric((nz_cricket_mean-total_cricket_mean)/(np.sqrt(nz_cricket_std+total_cricket_std)),errors="coerce")[0],3)

In [278]:
nz_cricket_away=nz[nz['Home']==False]
nz_win_away=len(nz_cricket_away[(nz_cricket_away["Team 1"]=="New Zealand")&(nz_cricket_away["Result"]=="won")])
other_lost_away=len(nz_cricket_away[(nz_cricket_away["Opposition"]=="New Zealand")&(nz_cricket_away["Result"]=='lost')])
nz_win_cricket_away=np.round((nz_win_away+other_lost_away)/(len(nz_cricket_away)),3)

In [279]:
nz_cricket_home=nz[nz["Home"]==True]
nz_win_home=len(nz_cricket_home[(nz_cricket_home["Team 1"]=="New Zealand")&(nz_cricket_home["Result"]=="won")])
other_lost_home=len(nz_cricket_home[(nz_cricket_home["Opposition"]=="New Zealand")&(nz_cricket_home["Result"]=='lost')])
nz_win_cricket_home=round((nz_win_home+other_lost_home)/(len(nz_cricket_home)),3)

In [280]:
nz_results.loc[0]=[np.round(cricket_z_score,3),nz_cricket_win_percent,nz_tie_percent,len(nz_cricket_home),nz_win_cricket_home,nz_win_cricket_away,'Cricket']

In [282]:
nz_results

,Z score,Win percentage,Tie percentage,Home Matches,Home Win Percentage,Away win Percentage,Sport
0,0.391000,0.596,0.010,17,0.882,0.537,Cricket
1,13.098028,0.754,0.034,233,0.830,0.699,Rugby
2,2.960000,0.412,0.175,128,0.570,0.343,Football


# Computing how significant RPO for NZ team is

In [128]:
# Combing all data about New Zealand Runs per over as Team 1 and as opposition
nz_rpo=np.concatenate((nz[nz["Team 1"]=="New Zealand"]["RPO"],nz[nz["Opposition"]=="New Zealand"]["Opposition RPO"]))
nz_mean_rpo=nz_rpo.mean()
nz_std_rpo=nz_rpo.std()
v1=nz_std_rpo**2/len(nz_rpo)

In [129]:
# Combinging all data about RPO as team 1 and RPO as opposition
all_rpo=np.concatenate((cricket_df["RPO"],cricket_df["Opposition RPO"]))
mean_rpo=all_rpo.mean()
std_rpo=all_rpo.std()
v2=std_rpo**2/len(all_rpo)
rpo_zscore=(nz_mean_rpo-mean_rpo)/(np.sqrt(v1+v2))

# Start of Rugby Analysis

In [289]:
# Loading data about rugby results
rugby_df=pd.read_csv('rugby_results.csv')
rugby_df=rugby_df.drop(['competition','stadium','city','neutral'],axis=1)
nz_rugby=rugby_df[(rugby_df['home_team']=="New Zealand")|(rugby_df['away_team']=="New Zealand")]

In [290]:
# Calculating the percentage of wins as away team
nz_away_win=nz_rugby[(nz_rugby["away_team"]=="New Zealand")&(nz_rugby["away_score"]>nz_rugby["home_score"])]
# Calculating the percentage of wins as home team
nz_home_win=nz_rugby[(nz_rugby['home_team']=="New Zealand")&(nz_rugby['home_score']>nz_rugby['away_score'])]
# Final percentage of wins for NZ team
nz_win_rugby=round((len(nz_away_win)+len(nz_home_win))/len(nz_rugby),3)

In [291]:
# Select when matches is played in New Zealand
nz_home_rugby=nz_rugby[nz_rugby['country']=="New Zealand"]
nz_win_home_rugby=round(len(nz_home_rugby[nz_home_rugby["home_score"]>nz_home_rugby['away_score']])/(len(nz_home_rugby)),2)

In [292]:
# Combining Scores for Home team and away team for New Zealand
nz_rugby_score=np.concatenate((nz_rugby[nz_rugby["home_team"]=="New Zealand"]["home_score"],nz_rugby[nz_rugby["away_team"]=="New Zealand"]['away_score']))
nz_mean_rugby=nz_rugby_score.mean()
nz_std_rugby=nz_rugby_score.std()**2/len(nz_rugby)

In [293]:
# Combing scores for all Home teams and all away teams results
all_score_rugby=np.concatenate((rugby_df["home_score"],rugby_df["away_score"]))
all_mean_rugby=all_score_rugby.mean()
all_std_rugby=all_score_rugby.std()**2/len(all_score_rugby)

In [294]:
rugby_z_score=np.round((nz_mean_rugby-all_mean_rugby)/(np.sqrt(nz_std_rugby+all_std_rugby)),3)

In [295]:
# Percentage of matches that NZ tied
tied_rugby=round(len(nz_rugby[nz_rugby['home_score']==nz_rugby['away_score']])/len(nz_rugby),3)

In [296]:
# Percentage of matches not played in NZ that we win
nz_away=nz_rugby[nz_rugby["country"]!='New Zealand']
num_away=len(nz_away)
home_not_nz=len(nz_away[(nz_away["home_team"]=="New Zealand")&(nz_away["home_score"]>nz_away['away_score'])])
away_not_nz=len(nz_away[(nz_away["away_team"]=="New Zealand")&(nz_away["away_score"]>nz_away['home_score'])])
nz_away_win_rugby=round((home_not_nz+away_not_nz)/num_away,3)

In [298]:
nz_results.loc[1]=[rugby_z_score,nz_win_rugby,tied_rugby,len(nz_home_rugby),nz_win_home_rugby,nz_away_win_rugby,'Rugby']

In [299]:
nz_results

,Z score,Win percentage,Tie percentage,Home Matches,Home Win Percentage,Away win Percentage,Sport
0,0.391,0.596,0.010,17,0.882,0.537,Cricket
1,13.098,0.754,0.034,233,0.830,0.699,Rugby
2,2.960,0.412,0.175,128,0.570,0.343,Football


## Assessing statistics for worldcup matches

In [130]:
# Filtering for only world cup matches
worldcup_df=rugby_df[rugby_df["world_cup"]==True]
nz_worldcup=worldcup_df[(worldcup_df["home_team"]=="New Zealand")|(worldcup_df["away_team"]=="New Zealand")]

In [134]:
# Percentage of matches played by NZ that we win in the world cup
nz_worldcup_win_percentage=np.round((len(nz_worldcup[(nz_worldcup['home_team']=="New Zealand")&(nz_worldcup["home_score"]>nz_worldcup["away_score"])])+len(nz_worldcup[(nz_worldcup['away_team']=="New Zealand")&(nz_worldcup["away_score"]>nz_worldcup["home_score"])]))/(len(nz_worldcup)),3)

# Start of football analysis

In [137]:
# Reading dataset for football results
football_df=pd.read_csv("football_results.csv")
# Dropping irrelevant rows from data
football_df=football_df.drop(['city','neutral'],axis=1)

In [138]:
# Select data for all New Zealand results
nz_football=football_df[(football_df["home_team"]=="New Zealand")|(football_df["away_team"]=="New Zealand")]

In [28]:
# Results for games played in New Zealand
nz_home_football=nz_football[nz_football['country']=="New Zealand"]
nz_football=nz_football.dropna()

In [218]:
# Win percentage when NZ team plays in New Zealand
nz_home_win=np.round(len(nz_home_football[(nz_home_football["home_score"]>nz_home_football["away_score"])])/len(nz_home_football),3)

In [220]:
# Win percentage playing outside of New Zealand
nz_home_notnz_win=nz_football[(nz_football["country"]!="New Zealand")&(nz_football["home_team"]=="New Zealand")&(nz_football["home_score"]>nz_football["away_score"])]
nz_away_win=nz_football[(nz_football["away_team"]=="New Zealand")&(nz_football["away_score"]>nz_football["home_score"])]
nz_away_win_percent=round((len(nz_home_notnz_win)+len(nz_away_win))/len(nz_football[nz_football["country"]!="New Zealand"]),3)

In [269]:
nz_football_win_percentage=np.round((len(nz_football[(nz_football["home_team"]=="New Zealand")&(nz_football["home_score"]>nz_football["away_score"])])+len(nz_football[(nz_football["away_team"]=="New Zealand")&(nz_football["away_score"]>nz_football["home_score"])]))/len(nz_football),3)

In [270]:
nz_football_win_percentage

np.float64(0.412)

In [223]:
# Computing mean score for NZ team
nz_football_home=nz_football[nz_football["home_team"]=="New Zealand"]['home_score'].mean()
nz_away=nz_football[nz_football["away_team"]=="New Zealand"]["away_score"].mean()
nz_football_points=round((nz_football_home+nz_away)/2,3)

In [224]:
# Combine all Scores for New Zealand teams
nz_football_score_df=pd.DataFrame(np.concatenate((nz_football[nz_football["home_team"]=="New Zealand"]["home_score"],nz_football[nz_football["away_team"]=="New Zealand"]["away_score"]))).dropna()
nz_football_mean=nz_football_score_df.mean()
nz_football_std=nz_football_score_df.std()**2/len(nz_football_score_df)

In [225]:
# Combine all scores for teams in Football
all_football_df=pd.DataFrame(np.concatenate((football_df["home_score"],football_df["away_score"])))
all_football_df=all_football_df.dropna()
all_football_mean=all_football_df.mean()
all_football_std=all_football_df.std()**2/len(all_football_df)

In [226]:
# Finding the Z score of NZ teams compared to all teams
football_z_score=np.round((nz_football_mean-all_football_mean)/(np.sqrt(nz_football_std+all_football_std))[0],3)

In [274]:
# Percentage of games that NZ football team ties
nz_tied_football=np.round((len(nz_football[nz_football['home_score']==nz_football["away_score"]]))/(len(nz_football)),3)

In [271]:
# Adding NZ records to results dataframe. [mean points, win percentage, tied percentage,Home matches,home win percent, away win percent ]
nz_results.loc[2]=[football_z_score[0],nz_football_win_percentage,nz_tied_football,len(nz_home_football),nz_home_win,nz_away_win_percent,"Football"]

In [273]:
nz_results.to_csv("nz_results.csv",index=False)

## Analysis of NZ performance in FIFA events

In [92]:
# Get all FIFA related records
nz_fifa=nz_football[nz_football["tournament"].str.contains("FIFA",regex=True)]

In [234]:
# Percentage of FIFA matches NZ won
nz_fifa_win=(len(nz_fifa[(nz_fifa["home_team"]=="New Zealand")&(nz_fifa["home_score"]>nz_fifa["away_score"])])+len(nz_fifa[(nz_fifa["away_team"]=="New Zealand")&(nz_fifa["away_score"]>nz_fifa["home_score"])]))/len(nz_fifa)

In [235]:
nz_fifa_win

0.5157894736842106

In [87]:
nz_results["Mean score"]

0     0.391362
1    13.098028
2        2.963
Name: Mean score, dtype: object

In [ ]:
# Analysis from all sports countries we lost to most